In [17]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

2026-08-23 08:24:23 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/mmm_tdc/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Introducción

 Se obtendrán el número de llamadas realizadas por BPO ofreciendo tarjeta de crédito.

In [2]:
# Nombre columnas
sql = """
DESCRIBE resultados_vspc_canales.bpo_gestionados
"""
nombre_columnas = helper.obtener_dataframe(sql)
nombre_columnas

2026-08-19 10:17:56 - [INFO] - Transcurrido: 1787152677, Tiempo de Refresco = 1000


----------------------------------------------------------------------------------------------
  i    tipo                    nombre                     estado     hora_inicio   duracion   
----------------------------------------------------------------------------------------------
 1/1 DATAFRAME resultados_vspc_canales.bpo_gestionados   ejecutando   10:17:56 AM             

2026-08-19 10:17:57 - [INFO] - 41 filas, 3 columnas, 00:00.6 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME resultados_vspc_canales.bpo_gestionados   finalizado   10:17:56 AM     00:00.7 
----------------------------------------------------------------------------------------------


,name,type,comment
0,f_entrega_informe,timestamp,
1,num_doc,bigint,
2,tipo_doc,smallint,
3,prod_ofrecido,string,
4,modelo,string,
5,reg_validos,string,
6,causal_no_valido,string,
7,desmonte,string,
8,gestionado,string,
9,f_gestionado,timestamp,


In [15]:
# Última ingestion
ult_ing_bpo = helper.obtener_ultima_ingestion("resultados_vspc_canales.bpo_gestionados")
ult_ing_bpo

2026-08-21 16:13:56 - [INFO] - Buscando fechas para resultados_vspc_canales.bpo_gestionados
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Pl

{'year': 2026, 'month': 8, 'day': 21}

In [16]:
# Obtener nombre de columnas de la tabla resultados_vspc_canales.bpo_gestionados

sql = f"""
SELECT {', '.join(nombre_columnas['name'].tolist())}
FROM resultados_vspc_canales.bpo_gestionados
WHERE YEAR = {ult_ing_bpo['year']}
  AND MONTH = {ult_ing_bpo['month']}
  AND ingestion_day = {ult_ing_bpo['day']}
"""
print(sql)


SELECT f_entrega_informe, num_doc, tipo_doc, prod_ofrecido, modelo, reg_validos, causal_no_valido, desmonte, gestionado, f_gestionado, hora_gestion, contactado, desc_contacto, tel_contacto, validez, causal_no_apto, venta_cantada, codase, num_doc_asesor, nivel_venta, aliado, id_venta, f_venta, canal_venta, tipo_contacto, causal_no_venta, prod_aceptado, monto, moneda, revocatoria_sms, revocatoria_email, revocatoria_televentas, no_compartir_info_terceros, ingestion_year, ingestion_month, ingestion_day, num_producto, franquicia, tel_valido, year, month
FROM resultados_vspc_canales.bpo_gestionados
WHERE YEAR = 2026
  AND MONTH = 8
  AND ingestion_day = 21



In [26]:
# Número de llamadas bpo ofreciendo una tarjeta de crédito
sql = """
SELECT to_date(f_gestionado) as f_ymd, count(*) as num_llamadas_bpo_gral
FROM resultados_vspc_canales.bpo_gestionados
WHERE YEAR BETWEEN 2024 and 2026
  AND MONTH BETWEEN 1 and 12
  AND ingestion_day BETWEEN 1 and 31
  AND prod_ofrecido IN ('TARJETA_CREDITO_PLATINUM_MC',
                        'TDC Joven',
                        'TARJETA_CREDITO_GOLD_AE',
                        'TARJETA_CREDITO',
                        'TARJETA_CREDITO_ORO_MC',
                        'TARJETA_CREDITO_ORO_VS',
                        'TDC',
                        'TARJETA_CREDITO_GREEN_AE',
                        'TARJETA_CREDITO_AMEX_LIBRE',
                        'TC MULTIFRANQUICIA',
                        'TDC Multifranquicia',
                        'TARJETA_CREDITO_AVIANCALIFEMILES_VS',
                        'TARJETA DE CREDITO',
                        'TARJETA_CREDITO_BLACK_MC',
                        'TDC Ideal',
                        'TARJETA_CREDITO_OFFICIALNUESTRASELECCION',
                        'TARJETA_CREDITO_PLATINUM_VS',
                        'TARJETA_CREDITO_JOVEN_MC',
                        'TARJETA_CREDITO_MASTER_UNICA',
                        'TARJETA_CREDITO_CLASICA_MC',
                        'TARJETA_CREDITO_PLATINUM_AE',
                        'TARJETA_CREDITO_IDEAL_MC',
                        'TARJETA_CREDITO_CLASICA_VS',
                        'TARJETA_CREDITO_BLUE_AE',
                        'TARJETA_CREDITO_INFINITE_VS')
  AND modelo NOT IN ('Aumento de cupo TDC')
  AND contactado = "Contactado"
  AND desc_contacto = "Contacto efectivo"
GROUP BY 1
ORDER BY to_date(f_gestionado) DESC;
"""
df_num_llamadas_bpo_gral = helper.obtener_dataframe(sql)
df_num_llamadas_bpo_gral

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 2/2 DATAFRAME        descargando   08:31:00 AM             

2026-08-23 08:31:04 - [INFO] - 777 filas, 2 columnas, 00:03.9 consultando, 00:00.4 descargando, 00:00.0 convirtiendo


 2/2 DATAFRAME         finalizado   08:31:00 AM     00:04.5 
------------------------------------------------------------


,f_ymd,num_llamadas_bpo_gral
0,2026-08-20,3
1,2026-08-19,821
2,2026-08-18,382
3,2026-08-15,899
4,2026-08-14,1499
...,...,...
772,2024-01-04,8331
773,2024-01-03,7410
774,2024-01-02,6464
775,2023-12-30,133


In [27]:
# Escribir
df_num_llamadas_bpo_gral.to_excel('hist_data/num_llamadas_bpo_gral_20240101_20260819.xlsx', index=False) # MODIFICAR nombre archivo. Cambia según fecha de ejecución.

In [28]:
# Número de llamadas bpo ofreciendo una tarjeta de crédito a clientes que estuvieron en la experiencia digital y no la obtuvieron

sql = """
SELECT to_date(f_gestionado) as f_ymd, count(*) as num_llamadas_bpo_grla_rescate_dig
FROM resultados_vspc_canales.bpo_gestionados
WHERE YEAR BETWEEN 2024 and 2026
  AND MONTH BETWEEN 1 and 12
  AND ingestion_day BETWEEN 1 and 31
  AND prod_ofrecido IN ('TARJETA_CREDITO_PLATINUM_MC',
                        'TDC Joven',
                        'TARJETA_CREDITO_GOLD_AE',
                        'TARJETA_CREDITO',
                        'TARJETA_CREDITO_ORO_MC',
                        'TARJETA_CREDITO_ORO_VS',
                        'TDC',
                        'TARJETA_CREDITO_GREEN_AE',
                        'TARJETA_CREDITO_AMEX_LIBRE',
                        'TC MULTIFRANQUICIA',
                        'TDC Multifranquicia',
                        'TARJETA_CREDITO_AVIANCALIFEMILES_VS',
                        'TARJETA DE CREDITO',
                        'TARJETA_CREDITO_BLACK_MC',
                        'TDC Ideal',
                        'TARJETA_CREDITO_OFFICIALNUESTRASELECCION',
                        'TARJETA_CREDITO_PLATINUM_VS',
                        'TARJETA_CREDITO_JOVEN_MC',
                        'TARJETA_CREDITO_MASTER_UNICA',
                        'TARJETA_CREDITO_CLASICA_MC',
                        'TARJETA_CREDITO_PLATINUM_AE',
                        'TARJETA_CREDITO_IDEAL_MC',
                        'TARJETA_CREDITO_CLASICA_VS',
                        'TARJETA_CREDITO_BLUE_AE',
                        'TARJETA_CREDITO_INFINITE_VS')
  AND modelo IN ('Rescate Digital')
  AND contactado = "Contactado"
  AND desc_contacto = "Contacto efectivo"
GROUP BY 1
ORDER BY to_date(f_gestionado) DESC;
"""
df_num_llamadas_bpo_gral_rescate_dig = helper.obtener_dataframe(sql)
df_num_llamadas_bpo_gral_rescate_dig

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 3/3 DATAFRAME         ejecutando   08:33:38 AM             

2026-08-23 08:33:40 - [INFO] - 515 filas, 2 columnas, 00:02.0 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 3/3 DATAFRAME         finalizado   08:33:38 AM     00:02.1 
------------------------------------------------------------


,f_ymd,num_llamadas_bpo_grla_rescate_dig
0,2026-08-19,388
1,2026-08-18,202
2,2026-08-15,43
3,2026-08-14,52
4,2026-08-13,89
...,...,...
510,2024-01-09,2620
511,2024-01-06,218
512,2024-01-05,555
513,2024-01-04,589


In [29]:
# Escribir
df_num_llamadas_bpo_gral_rescate_dig.to_excel('hist_data/num_llamadas_bpo_gral_rescate_dig_20240101_20260819.xlsx', index=False) # MODIFICAR nombre archivo. Cambia según fecha de ejecución.